# Day 14 — Final Model & Prediction Pipeline

## EduPro Predictive Modeling

### Objective

Build the final predictive models for:

- EnrollmentCount
- CourseRevenue

Validate the selected models using the fixed Day 9 test set,
compare original versus engineered features, retrain the final
models on the complete dataset, generate final predictions, and
save reusable prediction pipelines.

### Final model candidates from Day 12

- EnrollmentCount → Random Forest
- CourseRevenue → Linear Regression

### Day 13 contribution

17 engineered features added to the original 9 modeling features.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import joblib

print("All libraries imported successfully.")

All libraries imported successfully.


In [2]:
# ============================================================
# DAY 14 PATH CONFIGURATION
# ============================================================

project_folder = Path(
    r"D:\Data Analytics Project\EduPro_Predictive_Modeling"
)

feature_folder = project_folder / "data" / "feature_data"
model_folder = project_folder / "data" / "model_data"
models_folder = project_folder / "models"

notebook_folder = project_folder / "notebooks"

# Day 9 input
day9_file_path = (
    model_folder /
    "EduPro_Day9_Train_Test_Splits.xlsx"
)

# Day 12 model evaluation reference
day12_file_path = (
    model_folder /
    "EduPro_Day12_Model_Evaluation.xlsx"
)

# Day 13 engineered features
day13_file_path = (
    feature_folder /
    "EduPro_Day13_Feature_Engineering.xlsx"
)

# Day 14 output
day14_file_path = (
    model_folder /
    "EduPro_Day14_Final_Model_Predictions.xlsx"
)

# Final model files
enrollment_model_path = (
    models_folder /
    "EduPro_Final_Enrollment_Model.joblib"
)

revenue_model_path = (
    models_folder /
    "EduPro_Final_Revenue_Model.joblib"
)

# Create folders if required
feature_folder.mkdir(parents=True, exist_ok=True)
model_folder.mkdir(parents=True, exist_ok=True)
models_folder.mkdir(parents=True, exist_ok=True)
notebook_folder.mkdir(parents=True, exist_ok=True)

print("Project folder:")
print(project_folder)

print("\nDay 9 input:")
print(day9_file_path)

print("\nDay 12 reference:")
print(day12_file_path)

print("\nDay 13 input:")
print(day13_file_path)

print("\nDay 14 output:")
print(day14_file_path)

print("\nModels folder:")
print(models_folder)

Project folder:
D:\Data Analytics Project\EduPro_Predictive_Modeling

Day 9 input:
D:\Data Analytics Project\EduPro_Predictive_Modeling\data\model_data\EduPro_Day9_Train_Test_Splits.xlsx

Day 12 reference:
D:\Data Analytics Project\EduPro_Predictive_Modeling\data\model_data\EduPro_Day12_Model_Evaluation.xlsx

Day 13 input:
D:\Data Analytics Project\EduPro_Predictive_Modeling\data\feature_data\EduPro_Day13_Feature_Engineering.xlsx

Day 14 output:
D:\Data Analytics Project\EduPro_Predictive_Modeling\data\model_data\EduPro_Day14_Final_Model_Predictions.xlsx

Models folder:
D:\Data Analytics Project\EduPro_Predictive_Modeling\models


In [3]:
# ============================================================
# INPUT FILE VALIDATION
# ============================================================

print("========== INPUT FILE VALIDATION ==========")

input_files = {
    "Day 9": day9_file_path,
    "Day 12": day12_file_path,
    "Day 13": day13_file_path
}

for name, path in input_files.items():
    print(f"\n{name} file:")
    print(path)
    print("Exists:", path.exists())

    if not path.exists():
        raise FileNotFoundError(
            f"{name} input file not found:\n{path}"
        )

print("\nAll required input files are available.")

========== INPUT FILE VALIDATION ==========

Day 9 file:
D:\Data Analytics Project\EduPro_Predictive_Modeling\data\model_data\EduPro_Day9_Train_Test_Splits.xlsx
Exists: True

Day 12 file:
D:\Data Analytics Project\EduPro_Predictive_Modeling\data\model_data\EduPro_Day12_Model_Evaluation.xlsx
Exists: True

Day 13 file:
D:\Data Analytics Project\EduPro_Predictive_Modeling\data\feature_data\EduPro_Day13_Feature_Engineering.xlsx
Exists: True

All required input files are available.


## Load Day 13 engineered data

In [5]:
# ============================================================
# LOAD DAY 13 ENGINEERED DATA
# ============================================================

engineered_data = pd.read_excel(
    day13_file_path,
    sheet_name="Engineered_Features"
)

print("Day 13 engineered dataset loaded successfully.")

print("\nShape:")
print(engineered_data.shape)

print("\nColumns:")
print(engineered_data.columns.tolist())

print("\nMissing values:")
print(engineered_data.isnull().sum().sum())

print("\nDuplicate CourseIDs:")
print(engineered_data["CourseID"].duplicated().sum())

Day 13 engineered dataset loaded successfully.

Shape:
(60, 32)

Columns:
['CourseID', 'CourseName', 'CourseCategory', 'CourseType', 'CourseLevel', 'CoursePrice', 'CourseDuration', 'CourseRating', 'TeacherID', 'TeacherName', 'TeacherRating', 'YearsOfExperience', 'Expertise', 'EnrollmentCount', 'CourseRevenue', 'PricePerDay', 'PriceSquared', 'DurationSquared', 'LogCoursePrice', 'LogCourseDuration', 'RatingGap', 'AverageRating', 'CourseQualityScore', 'ExperienceRatingScore', 'PriceRatingInteraction', 'PricePerRatingPoint', 'Category_Type', 'Category_Level', 'Type_Level', 'Category_Expertise', 'Level_Expertise', 'ExperienceBand']

Missing values:
0

Duplicate CourseIDs:
0


## Load Day 9 Split 

In [8]:
# ============================================================
# LOAD DAY 9 TRAIN / TEST SPLIT
# ============================================================

train_data = pd.read_excel(
    day9_file_path,
    sheet_name="Train_Data"
)

test_data = pd.read_excel(
    day9_file_path,
    sheet_name="Test_Data"
)

X_train_original = pd.read_excel(
    day9_file_path,
    sheet_name="X_Train"
)

X_test_original = pd.read_excel(
    day9_file_path,
    sheet_name="X_Test"
)

y_train = pd.read_excel(
    day9_file_path,
    sheet_name="Y_Train"
)

y_test = pd.read_excel(
    day9_file_path,
    sheet_name="Y_Test"
)

print("Day 9 train/test data loaded.")

print("\nTrain_Data shape:")
print(train_data.shape)

print("\nTest_Data shape:")
print(test_data.shape)

print("\nX_train shape:")
print(X_train_original.shape)

print("\nX_test shape:")
print(X_test_original.shape)

print("\ny_train shape:")
print(y_train.shape)

print("\ny_test shape:")
print(y_test.shape)

Day 9 train/test data loaded.

Train_Data shape:
(48, 15)

Test_Data shape:
(12, 15)

X_train shape:
(48, 9)

X_test shape:
(12, 9)

y_train shape:
(48, 2)

y_test shape:
(12, 2)


## Define features

In [9]:
# ============================================================
# FEATURE DEFINITIONS
# ============================================================

target_columns = [
    "EnrollmentCount",
    "CourseRevenue"
]

original_features = [
    "CourseCategory",
    "CourseType",
    "CourseLevel",
    "CoursePrice",
    "CourseDuration",
    "CourseRating",
    "TeacherRating",
    "YearsOfExperience",
    "Expertise"
]

engineered_features = [
    "PricePerDay",
    "PriceSquared",
    "DurationSquared",
    "LogCoursePrice",
    "LogCourseDuration",
    "RatingGap",
    "AverageRating",
    "CourseQualityScore",
    "ExperienceRatingScore",
    "PriceRatingInteraction",
    "PricePerRatingPoint",
    "Category_Type",
    "Category_Level",
    "Type_Level",
    "Category_Expertise",
    "Level_Expertise",
    "ExperienceBand"
]

all_modeling_features = (
    original_features +
    engineered_features
)

print("Target columns:")
print(target_columns)

print("\nOriginal features:", len(original_features))
print(original_features)

print("\nEngineered features:", len(engineered_features))
print(engineered_features)

print("\nTotal modeling features:", len(all_modeling_features))

Target columns:
['EnrollmentCount', 'CourseRevenue']

Original features: 9
['CourseCategory', 'CourseType', 'CourseLevel', 'CoursePrice', 'CourseDuration', 'CourseRating', 'TeacherRating', 'YearsOfExperience', 'Expertise']

Engineered features: 17
['PricePerDay', 'PriceSquared', 'DurationSquared', 'LogCoursePrice', 'LogCourseDuration', 'RatingGap', 'AverageRating', 'CourseQualityScore', 'ExperienceRatingScore', 'PriceRatingInteraction', 'PricePerRatingPoint', 'Category_Type', 'Category_Level', 'Type_Level', 'Category_Expertise', 'Level_Expertise', 'ExperienceBand']

Total modeling features: 26


## Reusable feature-engineering function

In [10]:
# ============================================================
# REUSABLE FEATURE ENGINEERING FUNCTION
# ============================================================

def create_engineered_features(df):
    """
    Apply the same feature engineering logic used in Day 13.
    """

    data = df.copy()

    # --------------------------------------------------------
    # Price / duration features
    # --------------------------------------------------------

    data["PricePerDay"] = (
        data["CoursePrice"] /
        data["CourseDuration"].replace(0, np.nan)
    )

    data["PricePerDay"] = (
        data["PricePerDay"]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    data["PriceSquared"] = (
        data["CoursePrice"] ** 2
    )

    data["DurationSquared"] = (
        data["CourseDuration"] ** 2
    )

    data["LogCoursePrice"] = np.log1p(
        data["CoursePrice"]
    )

    data["LogCourseDuration"] = np.log1p(
        data["CourseDuration"]
    )

    # --------------------------------------------------------
    # Quality / rating features
    # --------------------------------------------------------

    data["RatingGap"] = (
        data["CourseRating"] -
        data["TeacherRating"]
    )

    data["AverageRating"] = (
        data["CourseRating"] +
        data["TeacherRating"]
    ) / 2

    data["CourseQualityScore"] = (
        data["CourseRating"] *
        data["TeacherRating"]
    )

    data["ExperienceRatingScore"] = (
        data["YearsOfExperience"] *
        data["TeacherRating"]
    )

    # --------------------------------------------------------
    # Price / quality interaction
    # --------------------------------------------------------

    data["PriceRatingInteraction"] = (
        data["CoursePrice"] *
        data["AverageRating"]
    )

    data["PricePerRatingPoint"] = (
        data["CoursePrice"] /
        data["AverageRating"].replace(0, np.nan)
    )

    data["PricePerRatingPoint"] = (
        data["PricePerRatingPoint"]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    # --------------------------------------------------------
    # Categorical interactions
    # --------------------------------------------------------

    data["Category_Type"] = (
        data["CourseCategory"].astype(str)
        + "_"
        + data["CourseType"].astype(str)
    )

    data["Category_Level"] = (
        data["CourseCategory"].astype(str)
        + "_"
        + data["CourseLevel"].astype(str)
    )

    data["Type_Level"] = (
        data["CourseType"].astype(str)
        + "_"
        + data["CourseLevel"].astype(str)
    )

    data["Category_Expertise"] = (
        data["CourseCategory"].astype(str)
        + "_"
        + data["Expertise"].astype(str)
    )

    data["Level_Expertise"] = (
        data["CourseLevel"].astype(str)
        + "_"
        + data["Expertise"].astype(str)
    )

    # --------------------------------------------------------
    # Experience band
    # --------------------------------------------------------

    data["ExperienceBand"] = pd.cut(
        data["YearsOfExperience"],
        bins=[-np.inf, 5, 10, 20, np.inf],
        labels=[
            "Early",
            "Developing",
            "Experienced",
            "Highly_Experienced"
        ]
    )

    return data


print("Reusable feature engineering function created.")

Reusable feature engineering function created.


## Recreate features from original data

In [11]:
# ============================================================
# RECREATE ENGINEERED FEATURES
# ============================================================

recreated_data = create_engineered_features(
    train_data.append(test_data, ignore_index=True)
    if hasattr(train_data, "append")
    else pd.concat(
        [train_data, test_data],
        ignore_index=True
    )
)

print("Recreated engineered dataset shape:")
print(recreated_data.shape)

print("\nMissing values:")
print(recreated_data.isnull().sum().sum())

print("\nDuplicate CourseIDs:")
print(recreated_data["CourseID"].duplicated().sum())

Recreated engineered dataset shape:
(60, 32)

Missing values:
0

Duplicate CourseIDs:
0


## Better source validation

In [14]:
# ============================================================
# CELL 10 — DAY 13 VS RECREATED FEATURE VALIDATION
# ============================================================

day13_sorted = (
    engineered_data
    .sort_values("CourseID")
    .reset_index(drop=True)
)

recreated_sorted = (
    recreated_data
    .sort_values("CourseID")
    .reset_index(drop=True)
)

comparison_columns = engineered_features

print("========== FEATURE REPRODUCIBILITY CHECK ==========")

print("\nDay 13 rows:")
print(len(day13_sorted))

print("\nRecreated rows:")
print(len(recreated_sorted))

# ------------------------------------------------------------
# 1. Check row counts
# ------------------------------------------------------------

rows_match = len(day13_sorted) == len(recreated_sorted)

# ------------------------------------------------------------
# 2. Check feature columns
# ------------------------------------------------------------

columns_match = (
    list(day13_sorted[comparison_columns].columns)
    == list(recreated_sorted[comparison_columns].columns)
)

# ------------------------------------------------------------
# 3. Identify numerical and categorical features
# ------------------------------------------------------------

numerical_features = [
    col
    for col in comparison_columns
    if pd.api.types.is_numeric_dtype(day13_sorted[col])
]

categorical_features = [
    col
    for col in comparison_columns
    if col not in numerical_features
]

print("\nNumerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

# ------------------------------------------------------------
# 4. Compare numerical features
# ------------------------------------------------------------

numerical_match = True

if numerical_features:
    numerical_match = np.allclose(
        day13_sorted[numerical_features].to_numpy(dtype=float),
        recreated_sorted[numerical_features].to_numpy(dtype=float),
        equal_nan=True
    )

# ------------------------------------------------------------
# 5. Compare categorical features
# ------------------------------------------------------------

categorical_match = True

if categorical_features:
    day13_categorical = (
        day13_sorted[categorical_features]
        .astype("string")
        .fillna("<NA>")
        .reset_index(drop=True)
    )

    recreated_categorical = (
        recreated_sorted[categorical_features]
        .astype("string")
        .fillna("<NA>")
        .reset_index(drop=True)
    )

    categorical_match = day13_categorical.equals(
        recreated_categorical
    )

# ------------------------------------------------------------
# 6. Overall feature reproducibility
# ------------------------------------------------------------

feature_match = (
    rows_match
    and columns_match
    and numerical_match
    and categorical_match
)

print("\n========== VALIDATION RESULTS ==========")

print("\nRows match:")
print(rows_match)

print("\nFeature columns match:")
print(columns_match)

print("\nNumerical features reproduced exactly:")
print(numerical_match)

print("\nCategorical features reproduced exactly:")
print(categorical_match)

print("\nEngineered features reproduced exactly:")
print(feature_match)

========== FEATURE REPRODUCIBILITY CHECK ==========

Day 13 rows:
60

Recreated rows:
60

Numerical features:
['PricePerDay', 'PriceSquared', 'DurationSquared', 'LogCoursePrice', 'LogCourseDuration', 'RatingGap', 'AverageRating', 'CourseQualityScore', 'ExperienceRatingScore', 'PriceRatingInteraction', 'PricePerRatingPoint']

Categorical features:
['Category_Type', 'Category_Level', 'Type_Level', 'Category_Expertise', 'Level_Expertise', 'ExperienceBand']

========== VALIDATION RESULTS ==========

Rows match:
True

Feature columns match:
True

Numerical features reproduced exactly:
True

Categorical features reproduced exactly:
True

Engineered features reproduced exactly:
True


## Prepared engineered train/test datsets

In [15]:
# ============================================================
# PREPARE ENGINEERED TRAIN / TEST DATA
# ============================================================

train_ids = train_data["CourseID"].tolist()
test_ids = test_data["CourseID"].tolist()

engineered_train = engineered_data[
    engineered_data["CourseID"].isin(train_ids)
].copy()

engineered_test = engineered_data[
    engineered_data["CourseID"].isin(test_ids)
].copy()

# Preserve Day 9 order
engineered_train = (
    engineered_train
    .set_index("CourseID")
    .loc[train_ids]
    .reset_index()
)

engineered_test = (
    engineered_test
    .set_index("CourseID")
    .loc[test_ids]
    .reset_index()
)

X_train_engineered = engineered_train[
    all_modeling_features
].copy()

X_test_engineered = engineered_test[
    all_modeling_features
].copy()

y_train_engineered = engineered_train[
    target_columns
].copy()

y_test_engineered = engineered_test[
    target_columns
].copy()

print("Engineered train shape:")
print(X_train_engineered.shape)

print("\nEngineered test shape:")
print(X_test_engineered.shape)

print("\nEngineered training targets:")
print(y_train_engineered.shape)

print("\nEngineered testing targets:")
print(y_test_engineered.shape)

Engineered train shape:
(48, 26)

Engineered test shape:
(12, 26)

Engineered training targets:
(48, 2)

Engineered testing targets:
(12, 2)


## Define preprocessing 

In [16]:
# ============================================================
# PREPROCESSING CONFIGURATION
# ============================================================

categorical_features_original = [
    "CourseCategory",
    "CourseType",
    "CourseLevel",
    "Expertise"
]

numerical_features_original = [
    "CoursePrice",
    "CourseDuration",
    "CourseRating",
    "TeacherRating",
    "YearsOfExperience"
]

categorical_features_engineered = [
    "CourseCategory",
    "CourseType",
    "CourseLevel",
    "Expertise",
    "Category_Type",
    "Category_Level",
    "Type_Level",
    "Category_Expertise",
    "Level_Expertise",
    "ExperienceBand"
]

numerical_features_engineered = [
    "CoursePrice",
    "CourseDuration",
    "CourseRating",
    "TeacherRating",
    "YearsOfExperience",
    "PricePerDay",
    "PriceSquared",
    "DurationSquared",
    "LogCoursePrice",
    "LogCourseDuration",
    "RatingGap",
    "AverageRating",
    "CourseQualityScore",
    "ExperienceRatingScore",
    "PriceRatingInteraction",
    "PricePerRatingPoint"
]

print("Original categorical features:")
print(categorical_features_original)

print("\nOriginal numerical features:")
print(numerical_features_original)

print("\nEngineered categorical features:")
print(categorical_features_engineered)

print("\nEngineered numerical features:")
print(numerical_features_engineered)

Original categorical features:
['CourseCategory', 'CourseType', 'CourseLevel', 'Expertise']

Original numerical features:
['CoursePrice', 'CourseDuration', 'CourseRating', 'TeacherRating', 'YearsOfExperience']

Engineered categorical features:
['CourseCategory', 'CourseType', 'CourseLevel', 'Expertise', 'Category_Type', 'Category_Level', 'Type_Level', 'Category_Expertise', 'Level_Expertise', 'ExperienceBand']

Engineered numerical features:
['CoursePrice', 'CourseDuration', 'CourseRating', 'TeacherRating', 'YearsOfExperience', 'PricePerDay', 'PriceSquared', 'DurationSquared', 'LogCoursePrice', 'LogCourseDuration', 'RatingGap', 'AverageRating', 'CourseQualityScore', 'ExperienceRatingScore', 'PriceRatingInteraction', 'PricePerRatingPoint']


## Pipeline builder

In [17]:
# ============================================================
# PIPELINE BUILDER
# ============================================================

def build_pipeline(model, use_engineered_features=True):

    if use_engineered_features:

        categorical_features = (
            categorical_features_engineered
        )

        numerical_features = (
            numerical_features_engineered
        )

    else:

        categorical_features = (
            categorical_features_original
        )

        numerical_features = (
            numerical_features_original
        )

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ),
                categorical_features
            ),
            (
                "numerical",
                "passthrough",
                numerical_features
            )
        ],
        remainder="drop"
    )

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    return pipeline


print("Pipeline builder created.")

Pipeline builder created.


## Final candidate model

In [18]:
# ============================================================
# FINAL MODEL CANDIDATES
# ============================================================

enrollment_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1
)

revenue_model = LinearRegression()

print("Enrollment model:")
print(enrollment_model)

print("\nRevenue model:")
print(revenue_model)

Enrollment model:
RandomForestRegressor(n_estimators=200, random_state=42)

Revenue model:
LinearRegression()


## Original feature models

In [19]:
# ============================================================
# ORIGINAL FEATURE MODELS
# ============================================================

enrollment_original_pipeline = build_pipeline(
    RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1
    ),
    use_engineered_features=False
)

revenue_original_pipeline = build_pipeline(
    LinearRegression(),
    use_engineered_features=False
)

# Train
enrollment_original_pipeline.fit(
    X_train_original,
    y_train["EnrollmentCount"]
)

revenue_original_pipeline.fit(
    X_train_original,
    y_train["CourseRevenue"]
)

print("Original-feature models trained successfully.")

Original-feature models trained successfully.


## Engineered featured models

In [20]:
# ============================================================
# ENGINEERED FEATURE MODELS
# ============================================================

enrollment_engineered_pipeline = build_pipeline(
    RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1
    ),
    use_engineered_features=True
)

revenue_engineered_pipeline = build_pipeline(
    LinearRegression(),
    use_engineered_features=True
)

# Train
enrollment_engineered_pipeline.fit(
    X_train_engineered,
    y_train_engineered["EnrollmentCount"]
)

revenue_engineered_pipeline.fit(
    X_train_engineered,
    y_train_engineered["CourseRevenue"]
)

print("Engineered-feature models trained successfully.")

Engineered-feature models trained successfully.


## Evaluation helper

In [21]:
# ============================================================
# EVALUATION FUNCTION
# ============================================================

def evaluate_model(
    model,
    X_test,
    y_test,
    target,
    model_name,
    feature_set
):

    predictions = model.predict(X_test)

    mae = mean_absolute_error(
        y_test[target],
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test[target],
            predictions
        )
    )

    r2 = r2_score(
        y_test[target],
        predictions
    )

    return {
        "Target": target,
        "Model": model_name,
        "Feature_Set": feature_set,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }


print("Evaluation function created.")

Evaluation function created.


## Evaluate final candidates

In [22]:
# ============================================================
# EVALUATE ORIGINAL VS ENGINEERED
# ============================================================

results = []

# Enrollment - original
results.append(
    evaluate_model(
        enrollment_original_pipeline,
        X_test_original,
        y_test,
        "EnrollmentCount",
        "Random Forest",
        "Original Features"
    )
)

# Enrollment - engineered
results.append(
    evaluate_model(
        enrollment_engineered_pipeline,
        X_test_engineered,
        y_test_engineered,
        "EnrollmentCount",
        "Random Forest",
        "Engineered Features"
    )
)

# Revenue - original
results.append(
    evaluate_model(
        revenue_original_pipeline,
        X_test_original,
        y_test,
        "CourseRevenue",
        "Linear Regression",
        "Original Features"
    )
)

# Revenue - engineered
results.append(
    evaluate_model(
        revenue_engineered_pipeline,
        X_test_engineered,
        y_test_engineered,
        "CourseRevenue",
        "Linear Regression",
        "Engineered Features"
    )
)

final_candidate_results = pd.DataFrame(results)

print(final_candidate_results.round(4))

            Target              Model          Feature_Set        MAE  \
0  EnrollmentCount      Random Forest    Original Features    12.6538   
1  EnrollmentCount      Random Forest  Engineered Features    12.1971   
2    CourseRevenue  Linear Regression    Original Features  2725.9923   
3    CourseRevenue  Linear Regression  Engineered Features  6761.1810   

         RMSE      R2  
0     13.5396 -0.3847  
1     13.2711 -0.3303  
2   3542.4712  0.9881  
3  14153.5212  0.8093  


## Select final model

In [23]:
# ============================================================
# SELECT FINAL MODEL FOR EACH TARGET
# ============================================================

selected_rows = []

for target in target_columns:

    target_results = (
        final_candidate_results[
            final_candidate_results["Target"] == target
        ]
        .sort_values(
            by=["MAE", "RMSE"],
            ascending=True
        )
    )

    best_row = target_results.iloc[0]

    selected_rows.append(best_row)

final_model_selection = pd.DataFrame(
    selected_rows
).reset_index(drop=True)

print("========== FINAL MODEL SELECTION ==========")
print(final_model_selection.round(4))

========== FINAL MODEL SELECTION ==========
            Target              Model          Feature_Set        MAE  \
0  EnrollmentCount      Random Forest  Engineered Features    12.1971   
1    CourseRevenue  Linear Regression    Original Features  2725.9923   

        RMSE      R2  
0    13.2711 -0.3303  
1  3542.4712  0.9881  


## Display final decision

In [24]:
# ============================================================
# FINAL MODEL DECISION
# ============================================================

for _, row in final_model_selection.iterrows():

    print(
        f"{row['Target']} -> "
        f"{row['Model']} + "
        f"{row['Feature_Set']}"
    )

EnrollmentCount -> Random Forest + Engineered Features
CourseRevenue -> Linear Regression + Original Features


## Determine selected pipelines 

In [25]:
# ============================================================
# ASSIGN SELECTED PIPELINES
# ============================================================

enrollment_selection = final_model_selection[
    final_model_selection["Target"] == "EnrollmentCount"
].iloc[0]

revenue_selection = final_model_selection[
    final_model_selection["Target"] == "CourseRevenue"
].iloc[0]


if enrollment_selection["Feature_Set"] == "Engineered Features":
    final_enrollment_pipeline = enrollment_engineered_pipeline
    final_enrollment_features = all_modeling_features
else:
    final_enrollment_pipeline = enrollment_original_pipeline
    final_enrollment_features = original_features


if revenue_selection["Feature_Set"] == "Engineered Features":
    final_revenue_pipeline = revenue_engineered_pipeline
    final_revenue_features = all_modeling_features
else:
    final_revenue_pipeline = revenue_original_pipeline
    final_revenue_features = original_features


print("Final EnrollmentCount feature set:")
print(enrollment_selection["Feature_Set"])

print("\nFinal CourseRevenue feature set:")
print(revenue_selection["Feature_Set"])

Final EnrollmentCount feature set:
Engineered Features

Final CourseRevenue feature set:
Original Features


## Prepare complete datasets

In [26]:
# ============================================================
# PREPARE COMPLETE DATASET FOR FINAL TRAINING
# ============================================================

full_data = engineered_data.copy()

X_full_engineered = full_data[
    all_modeling_features
].copy()

y_full = full_data[
    target_columns
].copy()

X_full_original = full_data[
    original_features
].copy()

print("Complete dataset:")
print(full_data.shape)

print("\nFull engineered feature matrix:")
print(X_full_engineered.shape)

print("\nFull original feature matrix:")
print(X_full_original.shape)

print("\nFull target matrix:")
print(y_full.shape)

Complete dataset:
(60, 32)

Full engineered feature matrix:
(60, 26)

Full original feature matrix:
(60, 9)

Full target matrix:
(60, 2)


## Retrain final Enrollment model

In [27]:
# ============================================================
# RETRAIN FINAL ENROLLMENT MODEL ON ALL 60 COURSES
# ============================================================

if enrollment_selection["Feature_Set"] == "Engineered Features":

    final_enrollment_pipeline = build_pipeline(
        RandomForestRegressor(
            n_estimators=200,
            random_state=42,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1
        ),
        use_engineered_features=True
    )

    final_enrollment_pipeline.fit(
        X_full_engineered,
        y_full["EnrollmentCount"]
    )

else:

    final_enrollment_pipeline = build_pipeline(
        RandomForestRegressor(
            n_estimators=200,
            random_state=42,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1
        ),
        use_engineered_features=False
    )

    final_enrollment_pipeline.fit(
        X_full_original,
        y_full["EnrollmentCount"]
    )

print("Final EnrollmentCount model trained on all 60 courses.")

Final EnrollmentCount model trained on all 60 courses.


## Retrain final Revenue model

In [28]:
# ============================================================
# RETRAIN FINAL REVENUE MODEL ON ALL 60 COURSES
# ============================================================

if revenue_selection["Feature_Set"] == "Engineered Features":

    final_revenue_pipeline = build_pipeline(
        LinearRegression(),
        use_engineered_features=True
    )

    final_revenue_pipeline.fit(
        X_full_engineered,
        y_full["CourseRevenue"]
    )

else:

    final_revenue_pipeline = build_pipeline(
        LinearRegression(),
        use_engineered_features=False
    )

    final_revenue_pipeline.fit(
        X_full_original,
        y_full["CourseRevenue"]
    )

print("Final CourseRevenue model trained on all 60 courses.")

Final CourseRevenue model trained on all 60 courses.


## Genrate final predictions

In [29]:
# ============================================================
# GENERATE FINAL PREDICTIONS
# ============================================================

if enrollment_selection["Feature_Set"] == "Engineered Features":

    enrollment_predictions = (
        final_enrollment_pipeline.predict(
            X_full_engineered
        )
    )

else:

    enrollment_predictions = (
        final_enrollment_pipeline.predict(
            X_full_original
        )
    )


if revenue_selection["Feature_Set"] == "Engineered Features":

    revenue_predictions = (
        final_revenue_pipeline.predict(
            X_full_engineered
        )
    )

else:

    revenue_predictions = (
        final_revenue_pipeline.predict(
            X_full_original
        )
    )


final_predictions = full_data[
    [
        "CourseID",
        "CourseName",
        "CourseCategory",
        "CourseType",
        "CourseLevel",
        "CoursePrice",
        "TeacherID",
        "TeacherName",
        "EnrollmentCount",
        "CourseRevenue"
    ]
].copy()

final_predictions[
    "Predicted_EnrollmentCount"
] = enrollment_predictions

final_predictions[
    "Predicted_CourseRevenue"
] = revenue_predictions


# Enrollment is a count, so prevent negative predictions
final_predictions[
    "Predicted_EnrollmentCount"
] = (
    final_predictions[
        "Predicted_EnrollmentCount"
    ]
    .clip(lower=0)
    .round()
    .astype(int)
)

# Revenue cannot be negative
final_predictions[
    "Predicted_CourseRevenue"
] = (
    final_predictions[
        "Predicted_CourseRevenue"
    ]
    .clip(lower=0)
)

print("Final predictions generated.")

final_predictions.head(10)

Final predictions generated.


,CourseID,CourseName,CourseCategory,CourseType,CourseLevel,CoursePrice,TeacherID,TeacherName,EnrollmentCount,CourseRevenue,Predicted_EnrollmentCount,Predicted_CourseRevenue
0,CR00050,Computer Vision,Artificial Intelligence,Paid,Beginner,490.9,TC00040,Kimberly Miller,174,85416.6,172,82636.439393
1,CR00021,Data Analysis with Python,Data Science,Free,Intermediate,0.0,TC00016,David Carlson,196,0.0,190,1294.653422
2,CR00009,Web Design Fundamentals,Design,Free,Beginner,0.0,TC00051,John Obrien,155,0.0,163,841.850902
3,CR00022,Data Visualization,Data Science,Free,Beginner,0.0,TC00010,Frances Sanchez,177,0.0,178,2772.108353
4,CR00027,Neural Networks,Machine Learning,Free,Advanced,0.0,TC00036,Brenda Mclean,152,0.0,158,598.528544
5,CR00036,Agile Project Management,Project Management,Free,Beginner,0.0,TC00040,Kimberly Miller,177,0.0,173,1060.426743
6,CR00018,Social Media Marketing,Marketing,Free,Advanced,0.0,TC00053,Aaron Kirby,163,0.0,163,0.000000
7,CR00037,Scrum Essentials,Project Management,Free,Intermediate,0.0,TC00040,Kimberly Miller,153,0.0,157,0.000000
8,CR00060,Content Creation,Digital Marketing,Free,Beginner,0.0,TC00036,Brenda Mclean,165,0.0,165,0.000000
9,CR00053,React for Beginners,Web Development,Free,Advanced,0.0,TC00042,Yolanda Levine,163,0.0,163,528.678280


## Prediction error on full training data

In [30]:
# ============================================================
# FULL-DATA FIT DIAGNOSTICS
# ============================================================

full_fit_diagnostics = pd.DataFrame({
    "Target": [
        "EnrollmentCount",
        "CourseRevenue"
    ],
    "MAE_Training_Fit": [
        mean_absolute_error(
            y_full["EnrollmentCount"],
            enrollment_predictions
        ),
        mean_absolute_error(
            y_full["CourseRevenue"],
            revenue_predictions
        )
    ],
    "RMSE_Training_Fit": [
        np.sqrt(
            mean_squared_error(
                y_full["EnrollmentCount"],
                enrollment_predictions
            )
        ),
        np.sqrt(
            mean_squared_error(
                y_full["CourseRevenue"],
                revenue_predictions
            )
        )
    ],
    "R2_Training_Fit": [
        r2_score(
            y_full["EnrollmentCount"],
            enrollment_predictions
        ),
        r2_score(
            y_full["CourseRevenue"],
            revenue_predictions
        )
    ]
})

print(full_fit_diagnostics.round(4))

            Target  MAE_Training_Fit  RMSE_Training_Fit  R2_Training_Fit
0  EnrollmentCount            3.6837             4.5475           0.8659
1    CourseRevenue         1120.8705          1664.4238           0.9956


## Prediction pipeline function

In [31]:
# ============================================================
# REUSABLE FUTURE PREDICTION FUNCTION
# ============================================================

def predict_new_courses(new_course_data):
    """
    Generate EnrollmentCount and CourseRevenue predictions
    for new course records.

    Input:
        DataFrame containing the original course-level
        modeling features.

    Required columns:
        CourseCategory
        CourseType
        CourseLevel
        CoursePrice
        CourseDuration
        CourseRating
        TeacherRating
        YearsOfExperience
        Expertise
    """

    new_data = new_course_data.copy()

    # Apply Day 13 feature engineering
    new_engineered = create_engineered_features(
        new_data
    )

    # Enrollment prediction
    if enrollment_selection["Feature_Set"] == "Engineered Features":

        enrollment_input = new_engineered[
            all_modeling_features
        ]

        predicted_enrollment = (
            final_enrollment_pipeline.predict(
                enrollment_input
            )
        )

    else:

        enrollment_input = new_engineered[
            original_features
        ]

        predicted_enrollment = (
            final_enrollment_pipeline.predict(
                enrollment_input
            )
        )

    # Revenue prediction
    if revenue_selection["Feature_Set"] == "Engineered Features":

        revenue_input = new_engineered[
            all_modeling_features
        ]

        predicted_revenue = (
            final_revenue_pipeline.predict(
                revenue_input
            )
        )

    else:

        revenue_input = new_engineered[
            original_features
        ]

        predicted_revenue = (
            final_revenue_pipeline.predict(
                revenue_input
            )
        )

    result = new_data.copy()

    result[
        "Predicted_EnrollmentCount"
    ] = (
        np.clip(
            predicted_enrollment,
            0,
            None
        )
        .round()
        .astype(int)
    )

    result[
        "Predicted_CourseRevenue"
    ] = np.clip(
        predicted_revenue,
        0,
        None
    )

    return result


print("Reusable prediction pipeline created.")

Reusable prediction pipeline created.


## Test prediction pipeline

In [32]:
# ============================================================
# TEST REUSABLE PREDICTION PIPELINE
# ============================================================

sample_new_courses = full_data[
    original_features
].head(3).copy()

pipeline_test_predictions = predict_new_courses(
    sample_new_courses
)

print("Sample prediction pipeline output:")
display(pipeline_test_predictions)

Sample prediction pipeline output:


,CourseCategory,CourseType,CourseLevel,CoursePrice,CourseDuration,CourseRating,TeacherRating,YearsOfExperience,Expertise,Predicted_EnrollmentCount,Predicted_CourseRevenue
0,Artificial Intelligence,Paid,Beginner,490.9,7.55,4.55,4.58,24,Cybersecurity,172,82636.439393
1,Data Science,Free,Intermediate,0.0,1.20,3.60,2.92,1,Data Science,190,1294.653422
2,Design,Free,Beginner,0.0,48.19,4.51,1.77,2,Design,163,841.850902


## Save trained models

In [33]:
# ============================================================
# SAVE FINAL MODELS
# ============================================================

joblib.dump(
    final_enrollment_pipeline,
    enrollment_model_path
)

joblib.dump(
    final_revenue_pipeline,
    revenue_model_path
)

print("Final models saved successfully.")

print("\nEnrollment model:")
print(enrollment_model_path)

print("\nRevenue model:")
print(revenue_model_path)

print("\nEnrollment model exists:")
print(enrollment_model_path.exists())

print("\nRevenue model exists:")
print(revenue_model_path.exists())

Final models saved successfully.

Enrollment model:
D:\Data Analytics Project\EduPro_Predictive_Modeling\models\EduPro_Final_Enrollment_Model.joblib

Revenue model:
D:\Data Analytics Project\EduPro_Predictive_Modeling\models\EduPro_Final_Revenue_Model.joblib

Enrollment model exists:
True

Revenue model exists:
True


## Model Summary 

In [34]:
# ============================================================
# FINAL MODEL SUMMARY
# ============================================================

final_model_summary = pd.DataFrame({
    "Target": [
        "EnrollmentCount",
        "CourseRevenue"
    ],
    "Selected_Model": [
        enrollment_selection["Model"],
        revenue_selection["Model"]
    ],
    "Selected_Feature_Set": [
        enrollment_selection["Feature_Set"],
        revenue_selection["Feature_Set"]
    ],
    "Test_MAE": [
        enrollment_selection["MAE"],
        revenue_selection["MAE"]
    ],
    "Test_RMSE": [
        enrollment_selection["RMSE"],
        revenue_selection["RMSE"]
    ],
    "Test_R2": [
        enrollment_selection["R2"],
        revenue_selection["R2"]
    ]
})

print(final_model_summary.round(4))

            Target     Selected_Model Selected_Feature_Set   Test_MAE  \
0  EnrollmentCount      Random Forest  Engineered Features    12.1971   
1    CourseRevenue  Linear Regression    Original Features  2725.9923   

   Test_RMSE  Test_R2  
0    13.2711  -0.3303  
1  3542.4712   0.9881  


## Model configuration

In [35]:
# ============================================================
# MODEL CONFIGURATION
# ============================================================

model_configuration = pd.DataFrame({
    "Target": [
        "EnrollmentCount",
        "CourseRevenue"
    ],
    "Algorithm": [
        "Random Forest",
        "Linear Regression"
    ],
    "Random_State": [
        42,
        np.nan
    ],
    "Estimators": [
        200,
        np.nan
    ],
    "Feature_Set_Selected": [
        enrollment_selection["Feature_Set"],
        revenue_selection["Feature_Set"]
    ],
    "Training_Rows_Final": [
        len(full_data),
        len(full_data)
    ]
})

model_configuration

,Target,Algorithm,Random_State,Estimators,Feature_Set_Selected,Training_Rows_Final
0,EnrollmentCount,Random Forest,42.0,200.0,Engineered Features,60
1,CourseRevenue,Linear Regression,NaN,NaN,Original Features,60


## Prediction summary

In [36]:
# ============================================================
# PREDICTION SUMMARY
# ============================================================

prediction_summary = pd.DataFrame({
    "Metric": [
        "Number of courses",
        "Actual total enrollment",
        "Predicted total enrollment",
        "Actual total revenue",
        "Predicted total revenue",
        "Average predicted enrollment",
        "Average predicted revenue"
    ],
    "Value": [
        len(final_predictions),

        final_predictions[
            "EnrollmentCount"
        ].sum(),

        final_predictions[
            "Predicted_EnrollmentCount"
        ].sum(),

        final_predictions[
            "CourseRevenue"
        ].sum(),

        final_predictions[
            "Predicted_CourseRevenue"
        ].sum(),

        final_predictions[
            "Predicted_EnrollmentCount"
        ].mean(),

        final_predictions[
            "Predicted_CourseRevenue"
        ].mean()
    ]
})

prediction_summary

,Metric,Value
0,Number of courses,60.000000
1,Actual total enrollment,10000.000000
2,Predicted total enrollment,9996.000000
3,Actual total revenue,911323.470000
4,Predicted total revenue,925124.042802
5,Average predicted enrollment,166.600000
6,Average predicted revenue,15418.734047


## Feature documentation

In [37]:
# ============================================================
# FINAL FEATURE DOCUMENTATION
# ============================================================

feature_documentation = pd.DataFrame({
    "Feature": (
        original_features +
        engineered_features
    ),
    "Feature_Group": (
        ["Original"] * len(original_features) +
        ["Engineered"] * len(engineered_features)
    )
})

feature_documentation

,Feature,Feature_Group
0,CourseCategory,Original
1,CourseType,Original
2,CourseLevel,Original
3,CoursePrice,Original
4,CourseDuration,Original
5,CourseRating,Original
6,TeacherRating,Original
7,YearsOfExperience,Original
8,Expertise,Original
9,PricePerDay,Engineered


## Save day 14 workbook

In [38]:
# ============================================================
# SAVE DAY 14 RESULTS
# ============================================================

with pd.ExcelWriter(
    day14_file_path,
    engine="openpyxl"
) as writer:

    final_model_summary.to_excel(
        writer,
        sheet_name="Final_Model_Summary",
        index=False
    )

    final_candidate_results.to_excel(
        writer,
        sheet_name="Candidate_Comparison",
        index=False
    )

    final_predictions.to_excel(
        writer,
        sheet_name="Final_Predictions",
        index=False
    )

    prediction_summary.to_excel(
        writer,
        sheet_name="Prediction_Summary",
        index=False
    )

    model_configuration.to_excel(
        writer,
        sheet_name="Model_Configuration",
        index=False
    )

    feature_documentation.to_excel(
        writer,
        sheet_name="Feature_Documentation",
        index=False
    )

    full_fit_diagnostics.to_excel(
        writer,
        sheet_name="Training_Fit_Diagnostics",
        index=False
    )

print("Day 14 results saved successfully:")
print(day14_file_path)

Day 14 results saved successfully:
D:\Data Analytics Project\EduPro_Predictive_Modeling\data\model_data\EduPro_Day14_Final_Model_Predictions.xlsx


## Reload and validate

In [39]:
# ============================================================
# DAY 14 OUTPUT VALIDATION
# ============================================================

check_summary = pd.read_excel(
    day14_file_path,
    sheet_name="Final_Model_Summary"
)

check_predictions = pd.read_excel(
    day14_file_path,
    sheet_name="Final_Predictions"
)

check_candidates = pd.read_excel(
    day14_file_path,
    sheet_name="Candidate_Comparison"
)

print("==============================================")
print("DAY 14 FINAL MODEL VALIDATION")
print("==============================================")

print("\nFinal Model Summary:")
print(check_summary)

print("\nFinal Predictions shape:")
print(check_predictions.shape)

print("\nCandidate comparison shape:")
print(check_candidates.shape)

print("\nOutput file exists:")
print(day14_file_path.exists())

print("\nEnrollment model exists:")
print(enrollment_model_path.exists())

print("\nRevenue model exists:")
print(revenue_model_path.exists())

DAY 14 FINAL MODEL VALIDATION

Final Model Summary:
            Target     Selected_Model Selected_Feature_Set     Test_MAE  \
0  EnrollmentCount      Random Forest  Engineered Features    12.197083   
1    CourseRevenue  Linear Regression    Original Features  2725.992334   

     Test_RMSE   Test_R2  
0    13.271147 -0.330348  
1  3542.471175  0.988053  

Final Predictions shape:
(60, 12)

Candidate comparison shape:
(4, 6)

Output file exists:
True

Enrollment model exists:
True

Revenue model exists:
True


## Final integrity checks

In [40]:
# ============================================================
# FINAL INTEGRITY CHECKS
# ============================================================

assert len(final_predictions) == 60

assert (
    final_predictions["CourseID"].nunique()
    == 60
)

assert (
    final_predictions[
        "Predicted_EnrollmentCount"
    ].isnull().sum()
    == 0
)

assert (
    final_predictions[
        "Predicted_CourseRevenue"
    ].isnull().sum()
    == 0
)

assert (
    final_predictions[
        "Predicted_EnrollmentCount"
    ].ge(0).all()
)

assert (
    final_predictions[
        "Predicted_CourseRevenue"
    ].ge(0).all()
)

assert day14_file_path.exists()

assert enrollment_model_path.exists()

assert revenue_model_path.exists()

print("All Day 14 integrity checks passed.")

All Day 14 integrity checks passed.


## Final completion 

In [41]:
# ============================================================
# DAY 14 COMPLETION
# ============================================================

print("==============================================")
print("DAY 14 FINAL MODEL & PREDICTION PIPELINE")
print("COMPLETED")
print("==============================================")

print("\nInput — Day 13:")
print(day13_file_path)

print("\nReference — Day 12:")
print(day12_file_path)

print("\nValidation split — Day 9:")
print(day9_file_path)

print("\nFinal output:")
print(day14_file_path)

print("\nEnrollment model:")
print(enrollment_model_path)

print("\nRevenue model:")
print(revenue_model_path)

print("\nFinal models:")
print(
    final_model_summary[
        [
            "Target",
            "Selected_Model",
            "Selected_Feature_Set",
            "Test_MAE",
            "Test_RMSE",
            "Test_R2"
        ]
    ].round(4)
)

print("\nCourses predicted:")
print(len(final_predictions))

print("\n==============================================")
print("DAY 14 COMPLETED")
print("==============================================")

DAY 14 FINAL MODEL & PREDICTION PIPELINE
COMPLETED

Input — Day 13:
D:\Data Analytics Project\EduPro_Predictive_Modeling\data\feature_data\EduPro_Day13_Feature_Engineering.xlsx

Reference — Day 12:
D:\Data Analytics Project\EduPro_Predictive_Modeling\data\model_data\EduPro_Day12_Model_Evaluation.xlsx

Validation split — Day 9:
D:\Data Analytics Project\EduPro_Predictive_Modeling\data\model_data\EduPro_Day9_Train_Test_Splits.xlsx

Final output:
D:\Data Analytics Project\EduPro_Predictive_Modeling\data\model_data\EduPro_Day14_Final_Model_Predictions.xlsx

Enrollment model:
D:\Data Analytics Project\EduPro_Predictive_Modeling\models\EduPro_Final_Enrollment_Model.joblib

Revenue model:
D:\Data Analytics Project\EduPro_Predictive_Modeling\models\EduPro_Final_Revenue_Model.joblib

Final models:
            Target     Selected_Model Selected_Feature_Set   Test_MAE  \
0  EnrollmentCount      Random Forest  Engineered Features    12.1971   
1    CourseRevenue  Linear Regression    Original Feat